# モデル学習・評価ノートブック

このノートブックでは、顧客ごとの購入金額予測モデルを構築します。

## 問題設定
- 前日までの30日間のデータを使って向こう30日間の購入金額を予測
- 時系列交差検証によってモデルの性能を評価

In [ ]:
import gc
import warnings
from datetime import timedelta
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")

## データの読み込みと前処理

In [ ]:
def load_and_preprocess_data(file_path: str) -> pd.DataFrame:
    dtype = {
        "Invoice": "str",
        "StockCode": "str",
        "Description": "str",
        "Quantity": "int",
        "InvoiceDate": "str",
        "Price": "float",
        "Customer ID": "str",
        "Country": "str",
    }
    df = pd.read_csv(file_path, dtype=dtype)
    df.columns = [
        "invoice", "stock_code", "description", "quantity",
        "invoice_date", "price", "customer_id", "country"
    ]

    df["purchase_amount"] = df["price"] * df["quantity"]
    df["timestamp"] = pd.to_datetime(df["invoice_date"])
    df["date"] = df["timestamp"].dt.date.astype(str)
    del df["invoice_date"]
    gc.collect()

    # 以下のようなレコードを削除
    # - CustomerID が欠損している
    # - キャンセルされた
    # - purchase_amount がマイナス
    df = df[df["customer_id"].notna()].reset_index(drop=True)
    is_cancelled = df["invoice"].str.contains("C")
    df = df[~is_cancelled].reset_index(drop=True)
    df = df[df["purchase_amount"] > 0].reset_index(drop=True)

    print(f"件数: {len(df):,}")
    print(f"期間: {df["timestamp"].min()} ~ {df["timestamp"].max()}")
    print(f"顧客数: {df["customer_id"].nunique():,}")

    return df

In [ ]:
df = load_and_preprocess_data("online_retail_II.csv")
df.head()

## 特徴量・目的変数の作成

In [ ]:
def create_features(
    df: pd.DataFrame,
    prediction_date: str,
    feature_days: int = 90,
) -> pd.DataFrame:
    """
    指定した予測日(prediction_date)に対して、前日までの集計日数(feature_days)分のデータから特徴量を作成
    """
    # 特徴量集計の期間を指定
    pred_date = pd.to_datetime(prediction_date)
    end_date = pred_date - timedelta(days=1)
    start_date = end_date - timedelta(days=feature_days - 1)
    df_period = df[(df["timestamp"] >= start_date) & (df["timestamp"] <= end_date)]

    # 顧客ごとに集計
    features = aggregate(df_period)
    features["prediction_date"] = prediction_date

    # 顧客とinvoice, 顧客とdate それぞれで購入金額を集計
    group_cols = ["invoice", "date"]
    agg_col = "purchase_amount"
    for group_col in group_cols:
        agg_df = aggregate_2cols(
            df_period,
            agg_col=agg_col,
            group1_col="customer_id",
            group2_col=group_col,
        )
        features = features.merge(agg_df, on="customer_id", how="left")

    features["elapsed_days_from_first"] = (pred_date - pd.to_datetime(features["date_min"])).dt.days
    features["elapsed_days_from_last"] = (pred_date - pd.to_datetime(features["date_max"])).dt.days

    return features


def aggregate(df) -> pd.DataFrame:
    """顧客ごとに集計して特徴量を作成"""
    agg_df = df.groupby("customer_id").agg({
        "invoice": ["count", "nunique"],  # レコード数、Invoiceのnunique
        "stock_code": "nunique",
        "timestamp": "nunique",
        "date": ["nunique", "min", "max"],
        "country": [lambda x: x[::-1].value_counts().idxmax(), "nunique"],
        "purchase_amount": "sum",
    }).reset_index()

    # カラム名をフラット化
    agg_df.columns = [
        "_".join(col).strip() if col[1] else col[0]
        for col in agg_df.columns
    ]
    agg_df = agg_df.rename(columns={
        "invoice_count": "record_count",
        "country_<lambda_0>": "country_mode",
    })
    return agg_df


def aggregate_2cols(
    df,
    agg_col="purchase_amount",
    group1_col="customer_id",
    group2_col="invoice",
) -> pd.DataFrame:
    """customer_id, invoiceまたはdateごとの購入金額のmax, min, meanを計算"""
    agg = df.groupby([group1_col, group2_col])[agg_col].sum().reset_index()
    agg_df = agg.groupby(group1_col)[agg_col].agg(["min", "max", "mean"]).reset_index()
    agg_df.columns = [group1_col] + [
        f"{agg_col}_by_{group2_col}_{col}"
        for col in agg_df.columns
        if col != group1_col
    ]
    return agg_df


def get_cols(df) -> tuple[list[str], list[str]]:
    rm_cols_pattern = [
        "customer_id",
        "date_min",
        "date_max",
        "prediction_date",
        "target",
    ]
    rm_cols = [c for c in df.columns if any(pattern in c for pattern in rm_cols_pattern)]
    use_cols = [c for c in df.columns if c not in rm_cols]

    cat_cols_pattern = ["country_mode"]
    cat_cols = [c for c in df.columns if any(pattern in c for pattern in cat_cols_pattern)]

    return use_cols, cat_cols

In [ ]:
def create_target(
    df: pd.DataFrame,
    prediction_date: str,
    target_days: int = 30,
) -> pd.DataFrame:
    """
    予測日(prediction_date)から指定日数(target_days)分の購入金額の合計を目的変数として作成
    """
    # 目的変数集計の期間を指定
    start_date = pd.to_datetime(prediction_date)
    end_date = start_date + timedelta(days=target_days - 1)
    df_period = df[(df["timestamp"] >= start_date) & (df["timestamp"] <= end_date)].reset_index(drop=True)

    # 顧客ごとに集計
    target = df_period.groupby("customer_id")["purchase_amount"].sum().reset_index()
    target.columns = ["customer_id", "target"]
    target["prediction_date"] = prediction_date

    return target

In [ ]:
def create_dataset(df: pd.DataFrame, prediction_date: str) -> pd.DataFrame:
    """特徴量と目的変数を結合してデータセットを作成"""
    features = create_features(df=df, prediction_date=prediction_date)
    target = create_target(df, prediction_date)
    dataset = features.merge(target, on=["customer_id", "prediction_date"],how="inner")

    return dataset

## 交差検証用データセットの作成

In [ ]:
cv_folds = {
    "fold1": {
        "train": ["2011-03-01", "2011-04-01", "2011-05-01"],
        "valid": ["2011-06-01"]
    },
    "fold2": {
        "train": ["2011-04-01", "2011-05-01", "2011-06-01"],
        "valid": ["2011-07-01"]
    },
    "fold3": {
        "train": ["2011-05-01", "2011-06-01", "2011-07-01"],
        "valid": ["2011-08-01"]
    },
    "fold4": {
        "train": ["2011-06-01", "2011-07-01", "2011-08-01"],
        "valid": ["2011-09-01"]
    },
    "fold5": {
        "train": ["2011-07-01", "2011-08-01", "2011-09-01"],
        "valid": ["2011-10-01"]
    },
    "test": {
        "train": ["2011-08-01", "2011-09-01", "2011-10-01"],
        "valid": ["2011-11-01"]
    }
}

all_datasets = {}
for fold_name, dates in cv_folds.items():
    train_datasets = []
    for date_str in dates["train"]:
        ds = create_dataset(df, date_str)
        train_datasets.append(ds)

    valid_datasets = []
    for date_str in dates["valid"]:
        ds = create_dataset(df, date_str)
        valid_datasets.append(ds)

    train_df = pd.concat(train_datasets, ignore_index=True)
    valid_df = pd.concat(valid_datasets, ignore_index=True)

    all_datasets[fold_name] = {"train": train_df, "valid": valid_df}
    print(f"{fold_name}: train={len(train_df)}, valid={len(valid_df)}")

In [ ]:
all_datasets["fold1"]["train"].head()

In [ ]:
tmp_df = all_datasets["fold5"]["train"]
print(tmp_df.shape)

num_cols = tmp_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
num_cols = [c for c in num_cols if c not in  ['customer_id', 'target']]
num_cols = [c for c in num_cols if "_90d" in c]
use_words = ["nunique", "mean", "from"]
num_cols = [c for c in num_cols if any(word in c for word in use_words)]
num_cols = num_cols + ['target']
num_cols

In [ ]:
# 数値型のカラムのみを選択
numeric_df = tmp_df.select_dtypes(include=[np.number])

# targetとの相関係数を計算
correlations = numeric_df.corr()['target'].drop('target').sort_values(ascending=False)

print(f"targetとの相関係数（上位10件）:")
print("=" * 50)
print(correlations.head(10))

print(f"\ntargetとの相関係数（下位10件）:")
print("=" * 50)
print(correlations.tail(10))

## モデルの学習と評価

In [ ]:
def encode_data(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    ordinal_encoders: dict = {},
) -> tuple:
    """カテゴリ変数のエンコーディング"""
    use_cols, cat_cols = get_cols(train_df)

    # 学習データのエンコーディング
    if not ordinal_encoders:
        for col in cat_cols:
            oe = OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
            train_df[col] = oe.fit_transform(train_df[col].to_numpy().reshape(-1, 1))
            ordinal_encoders[col] = oe
            train_df[col] = train_df[col].astype("category")
    else:
        for col in cat_cols:
            train_df[col] = ordinal_encoders[col].transform(train_df[col].to_numpy().reshape(-1, 1))
            train_df[col] = train_df[col].astype("category")

    # 検証データのエンコーディング
    for col in cat_cols:
        valid_df[col] = ordinal_encoders[col].transform(valid_df[col].to_numpy().reshape(-1, 1))
        valid_df[col] = valid_df[col].astype("category")

    # 特徴量と目的変数に分割
    train_x = train_df[use_cols]
    train_y = train_df["target"]
    valid_x = valid_df[use_cols]
    valid_y = valid_df["target"]

    return train_x, train_y, valid_x, valid_y, ordinal_encoders

In [ ]:
def train_lightgbm(
    train_x: pd.DataFrame,
    train_y: pd.Series,
    valid_x: pd.DataFrame,
    valid_y: pd.Series,
) -> lgb.Booster:
    """LightGBMモデルを学習"""
    train_data = lgb.Dataset(train_x, label=train_y)
    valid_data = lgb.Dataset(valid_x, label=valid_y, reference=train_data)
    params = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbose": -1,
    }
    model = lgb.train(
        params,
        train_data,
        valid_sets=[train_data, valid_data],
        valid_names=["train", "valid"],
        num_boost_round=1000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100)
        ]
    )
    return model

In [ ]:
def get_previous_month_amount(
    df: pd.DataFrame,
    prediction_date: str,
    feature_days: int = 30,
) -> pd.DataFrame:
    """ベンチマーク用で前月の購入金額を顧客ごとに計算"""
    end_date = pd.to_datetime(prediction_date) - timedelta(days=1)
    start_date = end_date - timedelta(days=feature_days - 1)
    df_period = df[(df["timestamp"] >= start_date) & (df["timestamp"] <= end_date)].reset_index(drop=True)

    prev_month = df_period.groupby("customer_id")["purchase_amount"].sum().reset_index()
    prev_month.columns = ["customer_id", "prev_month_amount"]
    prev_month["prediction_date"] = prediction_date

    return prev_month

In [ ]:
scores = []
ordinal_encoder = None

model_artifacts = []
for fold_name in ["fold1", "fold2", "fold3", "fold4", "fold5"]:
    print(f"\n==== fold: {fold_name} ====")

    train_df = all_datasets[fold_name]["train"]
    valid_df = all_datasets[fold_name]["valid"]
    train_x, train_y, valid_x, valid_y, ordinal_encoders = encode_data(train_df, valid_df)

    # LightGBM
    model = train_lightgbm(train_x, train_y, valid_x, valid_y)
    y_pred = model.predict(valid_x, num_iteration=model.best_iteration)

    # ベンチマーク(前月購入金額)
    valid_date = cv_folds[fold_name]["valid"][0]
    prev_df = get_previous_month_amount(df, prediction_date=valid_date)
    prev_df = valid_df[["customer_id", "prediction_date"]].merge(
        prev_df, on=["customer_id", "prediction_date"], how="left"
    )
    y_prev = prev_df["prev_month_amount"].fillna(0)

    # RMSE
    rmse_prev = root_mean_squared_error(valid_y, y_prev)
    rmse = root_mean_squared_error(valid_y, y_pred)

    print(f"\n{fold_name} scores:")
    print(f"  RMSE(前月購入金額): {rmse_prev:.2f}")
    print(f"  RMSE(モデル): {rmse:.2f}")

    scores.append({
        "fold": fold_name,
        "rmse_prev": rmse_prev,
        "rmse": rmse,
    })
    model_artifacts.append({
        "model": model,
        "ordinal_encoders": ordinal_encoders,
        "use_cols": list(train_x.columns),
    })

score_df = pd.DataFrame(scores)
print(score_df)

print("\nSummary")
print(f"  RMSE(前月購入金額): {score_df['rmse_prev'].mean():.2f}")
print(f"  RMSE(モデル): {score_df['rmse'].mean():.2f}")

## 特徴量の重要度

In [ ]:
importance_data = []
for artifact in model_artifacts:
    model = artifact["model"]
    feature_cols = artifact["use_cols"]
    importance = model.feature_importance(importance_type="gain")
    for feature, imp in zip(feature_cols, importance, strict=True):
        importance_data.append({"feature": feature, "importance": imp})
importance_df = pd.DataFrame(importance_data)

top_n = 20
feature_stats = importance_df.groupby("feature")["importance"].agg(["mean"]).reset_index()
feature_stats = feature_stats.sort_values("mean", ascending=False).reset_index(drop=True).head(top_n)

plt.figure(figsize=(12, 8))
x_pos = range(len(feature_stats))
bars = plt.barh(x_pos, feature_stats["mean"], color='blue', alpha=0.5)
plt.yticks(x_pos, feature_stats["feature"])
plt.xlabel("Feature Importance", fontsize=14)
plt.tick_params(axis='both', labelsize=14)
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print(feature_stats[["feature", "mean"]].round(2))

## モデルとartifactsの保存

In [ ]:
model_artifacts

In [ ]:
Path("models").mkdir(parents=True, exist_ok=True)

base_path = "models/lgb.pkl"
for fold_idx, artifact in enumerate(model_artifacts):
    save_path = base_path.replace(".pkl", f"_fold{fold_idx}.pkl")
    joblib.dump(artifact, save_path)
    print(f"Saved to {save_path}")

## 推論

In [ ]:
def load_model_and_predict(prediction_date: str, model_path: str) -> pd.DataFrame:
    artifact = joblib.load(model_path)
    model = artifact["model"]
    ordinal_encoders = artifact["ordinal_encoders"]
    feature_cols = artifact["use_cols"]

    test_df = load_and_preprocess_data("online_retail_II.csv")

    windows = [30, 60, 90]
    for idx, window in enumerate(windows):
        tmp_df = create_features(
            df=test_df,
            prediction_date=prediction_date,
            feature_days=window,
        )
        tmp_df.columns = ["customer_id"] + [
            f"{col}_in_{window}d" for col in tmp_df.columns if col != "customer_id"
        ]
        if idx == 0:
            features = tmp_df
        else:
            features = pd.merge(features, tmp_df, how="outer", on="customer_id")

    use_cols, cat_cols = get_cols(features)
    for col in cat_cols:
        features[col] = ordinal_encoders[col].transform(
            features[col].to_numpy().reshape(-1, 1)
        )
        features[col] = features[col].astype("category")

    x_test = features[feature_cols]
    predictions = model.predict(x_test, num_iteration=model.best_iteration)

    result = features[["customer_id"]].copy()
    result["prediction_date"] = prediction_date
    result["predicted_amount"] = predictions
    # 回帰モデルの予測値がマイナスになることがあるため、0未満の値はクリップ
    result["predicted_amount_clipped"] = result["predicted_amount"].clip(lower=0)

    return result

In [ ]:
result = load_model_and_predict(
    prediction_date="2011-12-01",
    model_path="models/lgb_fold4.pkl",
)
print(result.head())